<a href="https://colab.research.google.com/github/ibrahimlawrence283-beep/AuraGuard/blob/main/notebooks/00_phase0_environment_and_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))






GPU Available: True
Device Name: Tesla T4


In [2]:
!pip install -q transformers datasets peft bitsandbytes accelerate wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.7 MB/s eta 0:00:00


In [3]:
from transformers import AutoTokenizer

model_id = "Qwen/Qwen2.5-1.5B-Instruct"  # Lightweight model perfect for rapid testing
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Sample risk assessment instruction
messages = [
    {"role": "system", "content": "You are an expert financial risk and claims assessment assistant."},
    {"role": "user", "content": "Assess the risk level of an auto claim with missing police report details and $15,000 in claimed damages."}
]

# Apply chat template
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("--- Formatted Chat Prompt ---")
print(formatted_prompt)

# Tokenize and check tensor shapes
inputs = tokenizer(formatted_prompt, return_tensors="pt")
print("\nInput IDs shape:", inputs["input_ids"].shape)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

--- Formatted Chat Prompt ---
<|im_start|>system
You are an expert financial risk and claims assessment assistant.<|im_end|>
<|im_start|>user
Assess the risk level of an auto claim with missing police report details and $15,000 in claimed damages.<|im_end|>
<|im_start|>assistant


Input IDs shape: torch.Size([1, 50])


In [4]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load lightweight model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)

print("\nModel memory footprint:", model.get_memory_footprint() / 1e6, "MB")

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model memory footprint: 1122.135552 MB


In [5]:
from datasets import load_dataset

# 1. Load the dataset from Hugging Face
print("Downloading dataset...")
dataset = load_dataset("Cleanlab/insurance-claims-extraction", split="train")
print(f"Total records in raw dataset: {len(dataset)}\n")

# 2. Define the ChatML mapping function
def format_to_chatml(example):
    header = example.get('header', {})
    policy = example.get('policy_details', {})

    # Construct a structured prompt
    user_prompt = (
        f"Analyze the following insurance claim for risk evaluation:\n"
        f"- Claim ID: {header.get('claim_id', 'N/A')}\n"
        f"- Coverage Type: {policy.get('coverage_type', 'N/A')}\n"
        f"- Reporting Channel: {header.get('channel', 'N/A')}\n"
        f"- Incident Date: {header.get('incident_date', 'N/A')}\n"
        f"- Report Date: {header.get('report_date', 'N/A')}"
    )

    # Target Assistant Output (Synthetic Ground Truth target)
    assistant_response = (
        f"### Risk Assessment Summary\n"
        f"**Claim ID:** {header.get('claim_id', 'N/A')}\n"
        f"**Risk Level:** Moderate\n"
        f"**Key Findings:** Reported via {header.get('channel', 'N/A')} for {policy.get('coverage_type', 'N/A')} coverage. "
        f"Filing timeline needs verification against policy effective date ({policy.get('effective_date', 'N/A')})."
    )

    # ChatML Format
    formatted_chat = [
        {"role": "system", "content": "You are an expert AI risk and insurance claims analyst."},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_response}
    ]

    return {"messages": formatted_chat}

# 3. Transform the dataset
print("Mapping to ChatML format...")
formatted_dataset = dataset.map(format_to_chatml)

# 4. Verification
print("\n--- Formatted ChatML Example ---")
print(formatted_dataset[0]['messages'])

README.md:   0%|          | 0.00/141 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


insurance_claims_extraction.csv:   0%|          | 0.00/68.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30 [00:00<?, ? examples/s]

Total records in raw dataset: 30

Mapping to ChatML format...


Map:   0%|          | 0/30 [00:00<?, ? examples/s]


--- Formatted ChatML Example ---
[{'content': 'You are an expert AI risk and insurance claims analyst.', 'role': 'system'}, {'content': 'Analyze the following insurance claim for risk evaluation:\n- Claim ID: N/A\n- Coverage Type: N/A\n- Reporting Channel: N/A\n- Incident Date: N/A\n- Report Date: N/A', 'role': 'user'}, {'content': '### Risk Assessment Summary\n**Claim ID:** N/A\n**Risk Level:** Moderate\n**Key Findings:** Reported via N/A for N/A coverage. Filing timeline needs verification against policy effective date (N/A).', 'role': 'assistant'}]


In [6]:
print("Dataset columns:", dataset.column_names)
print("\nFirst raw example:")
print(dataset[0])


Dataset columns: ['claim_text', 'ground_truth']

First raw example:
{'claim_text': "[Email received: 2024-04-22]\n\nSubject: Property claim from Jennifer Garcia—CLM-424063 (Shopping Center Incident)\n\nHi—this is Jennifer Garcia, and I’m writing because, honestly, I’m still kind of in shock about what happened this morning. They gave me this claim reference CLM-424063 when I called the help line earlier, but I figured I should also email so everything’s in writing—it’s just been a really stressful day, and I want to make sure nothing gets missed. Not sure how these things usually work, I guess you’ll get back to me? Anyway...\n\nI’m the owner on Policy POL-787532354, with property coverage for my home at 6650 Broadway in Georgetown (you probably have it already under the id 656048, it’s the single-family one built in ‘72—big front yard, needs new gutters, but that’s not the issue today). My policy’s only been active since December 20th of last year, I think it's valid through all of Ju

In [7]:
import ast
from datasets import load_dataset

# Load raw dataset
dataset = load_dataset("Cleanlab/insurance-claims-extraction", split="train")

def format_to_chatml_v3(example):
    claim_text = example['claim_text']

    # Safely evaluate string representation of dict to real dict
    try:
        gt = ast.literal_eval(example['ground_truth'])
    except:
        gt = {}

    header = gt.get('header', {})
    policy = gt.get('policy_details', {})
    incident = gt.get('incident_description', {})

    # Construct System & User Prompt
    user_prompt = f"Analyze the following raw insurance claim email and extract structured risk signals:\n\n{claim_text}"

    # Target Assistant Extraction Output
    assistant_response = (
        f"### Extracted Risk & Claim Summary\n"
        f"- **Claim ID:** {header.get('claim_id', 'N/A')}\n"
        f"- **Policy Number:** {policy.get('policy_number', 'N/A')}\n"
        f"- **Policyholder:** {header.get('reported_by', 'N/A')}\n"
        f"- **Incident Type:** {incident.get('incident_type', 'N/A').title()}\n"
        f"- **Coverage Type:** {policy.get('coverage_type', 'N/A')}\n"
        f"- **Police Report #:** {incident.get('police_report_number', 'N/A')}\n"
        f"- **Estimated Damage:** ${incident.get('estimated_damage_amount', 'N/A'):,}\n\n"
        f"**Risk Flag:** Claim filed on {header.get('report_date', 'N/A')} under policy effective since {policy.get('effective_date', 'N/A')}."
    )

    formatted_chat = [
        {"role": "system", "content": "You are an expert AI risk and insurance claims analyst."},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_response}
    ]

    return {"messages": formatted_chat}

# Map dataset
cleaned_dataset = dataset.map(format_to_chatml_v3)

print("--- Cleaned Instruction Pair (User Prompt) ---")
print(cleaned_dataset[0]['messages'][1]['content'][:300] + "...\n")

print("--- Cleaned Instruction Pair (Target Response) ---")
print(cleaned_dataset[0]['messages'][2]['content'])

Repo card metadata block was not found. Setting CardData to empty.


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

AttributeError: 'NoneType' object has no attribute 'get'

In [8]:
import ast
from datasets import load_dataset

# Load raw dataset
dataset = load_dataset("Cleanlab/insurance-claims-extraction", split="train")

def format_to_chatml_v3(example):
    claim_text = example.get('claim_text', '')

    # Safely evaluate string representation of dict to real dict
    try:
        gt = ast.literal_eval(example.get('ground_truth', '{}'))
        if not isinstance(gt, dict):
            gt = {}
    except Exception:
        gt = {}

    header = gt.get('header') or {}
    policy = gt.get('policy_details') or {}
    incident = gt.get('incident_description') or {}

    # Extract damage amount safely
    dmg = incident.get('estimated_damage_amount')
    dmg_str = f"${dmg:,}" if isinstance(dmg, (int, float)) else 'N/A'

    # Construct System & User Prompt
    user_prompt = f"Analyze the following raw insurance claim email and extract structured risk signals:\n\n{claim_text}"

    # Target Assistant Extraction Output
    assistant_response = (
        f"### Extracted Risk & Claim Summary\n"
        f"- **Claim ID:** {header.get('claim_id') or 'N/A'}\n"
        f"- **Policy Number:** {policy.get('policy_number') or 'N/A'}\n"
        f"- **Policyholder:** {header.get('reported_by') or 'N/A'}\n"
        f"- **Incident Type:** {str(incident.get('incident_type') or 'N/A').title()}\n"
        f"- **Coverage Type:** {policy.get('coverage_type') or 'N/A'}\n"
        f"- **Police Report #:** {incident.get('police_report_number') or 'N/A'}\n"
        f"- **Estimated Damage:** {dmg_str}\n\n"
        f"**Risk Flag:** Claim filed on {header.get('report_date') or 'N/A'} under policy effective since {policy.get('effective_date') or 'N/A'}."
    )

    formatted_chat = [
        {"role": "system", "content": "You are an expert AI risk and insurance claims analyst."},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_response}
    ]

    return {"messages": formatted_chat}

# Map dataset safely
cleaned_dataset = dataset.map(format_to_chatml_v3)

print("--- Cleaned Instruction Pair (User Prompt) ---")
print(cleaned_dataset[0]['messages'][1]['content'][:300] + "...\n")

print("--- Cleaned Instruction Pair (Target Response) ---")
print(cleaned_dataset[0]['messages'][2]['content'])

Repo card metadata block was not found. Setting CardData to empty.


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

--- Cleaned Instruction Pair (User Prompt) ---
Analyze the following raw insurance claim email and extract structured risk signals:

[Email received: 2024-04-22]

Subject: Property claim from Jennifer Garcia—CLM-424063 (Shopping Center Incident)

Hi—this is Jennifer Garcia, and I’m writing because, honestly, I’m still kind of in shock about what...

--- Cleaned Instruction Pair (Target Response) ---
### Extracted Risk & Claim Summary
- **Claim ID:** CLM-424063
- **Policy Number:** POL-787532354
- **Policyholder:** Jennifer Garcia
- **Incident Type:** Vandalism
- **Coverage Type:** Property
- **Police Report #:** PR-20250822-4864
- **Estimated Damage:** $5,834

**Risk Flag:** Claim filed on 2024-04-22 under policy effective since 2023-12-20.
